# AI 기업 분석 실습 — 재무·주가·뉴스 건별 AI 감성 분석 (3차 고도화)

금융위원회 재무/주식시세 API, 캔들차트, 뉴스 관련성·중복 제거, 기사별 긍정/부정 분석 및 AI 투자검토를 실습합니다.
모든 중요 단계에는 별도 `assert` 검증 셀이 있습니다. 실제 API 서버 호출은 별도 확인이 필요합니다.

## 0단계 — 실행 환경 / 패키지 설치

In [ ]:
# 처음 한 번만 설치하세요. 셀에서 아래 줄의 #만 지우고 실행합니다.
# %pip install -r requirements.txt

# 동일 폴더에 core.py / .env / requirements.txt 파일이 있어야 합니다.
from pathlib import Path
print('Jupyter 작업 폴더:', Path.cwd())
print('core.py 확인:', Path('core.py').exists())
print('.env 확인:', Path('.env').exists())

## 1단계 — `.env` 인증키 불러오기

기업 재무정보 API, 금융위원회 주식시세정보 API, OpenAI API, 선택 NewsAPI·SerpAPI 키를 불러옵니다.

In [ ]:
import os
import re
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display, Markdown
from core import collect_all, excel_bytes, ai_review, report_markdown

load_dotenv(dotenv_path=Path('.env'), override=True)

# 키는 화면에 출력하지 않습니다.
공통키 = os.getenv('DATA_GO_KR_API_KEY', '').strip() or os.getenv('DATA_GO_KR_KEY', '').strip()
재무키 = os.getenv('FINANCE_API_KEY', '').strip() or 공통키
주가키 = os.getenv('STOCK_API_KEY', '').strip() or 공통키
OpenAI키 = os.getenv('OPENAI_API_KEY', '').strip()
뉴스키 = os.getenv('NEWS_API_KEY', '').strip()
SerpAPI키 = os.getenv('SERPAPI_KEY', '').strip()
모델 = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')

기업명 = '삼성전자'  # 이 곳만 다른 기업명으로 변경해도 됩니다.
사업연도수 = 3

print('조회 기업:', 기업명)
print('재무 API 인증키:', '설정됨' if 재무키 else '미설정')
print('주가 API 인증키:', '설정됨' if 주가키 else '미설정')
print('OpenAI 인증키:', '설정됨' if OpenAI키 else '미설정')
print('NewsAPI 키:', '설정됨' if 뉴스키 else '미설정')
print('SerpAPI 키:', '설정됨' if SerpAPI키 else '미설정')
print('뉴스 시도 순서: NewsAPI → SerpAPI → Google 뉴스 RSS')

### 별도 검증 셀 — 환경변수 설정

이 셀은 **키 존재 여부만** 확인합니다. 실제 키가 승인되어 있는지는 다음 단계에서 확인합니다.

In [ ]:
assert isinstance(기업명, str) and 기업명.strip(), '기업명을 입력하세요.'
assert 재무키 and not 재무키.startswith('발급받은_'), '.env에 재무 API 키가 필요합니다.'
assert 주가키 and not 주가키.startswith('발급받은_'), '.env에 주가 API 키가 필요합니다.'
print('환경변수 검증 통과 (실제 API 인증 성공은 아직 확인되지 않음)')

## 2단계 — 기업명으로 재무·주가·뉴스 수집

각 데이터는 독립 수집합니다. 한 출처가 실패해도 다른 출처의 데이터를 사용할 수 있습니다.

In [ ]:
결과 = collect_all(
    company=기업명,
    finance_key=재무키,
    stock_key=주가키,
    year_count=사업연도수,
    news_key=뉴스키 or None,
    serpapi_key=SerpAPI키 or None,
)

재무표 = 결과['재무']
주가표 = 결과['주가']
뉴스표 = 결과['뉴스']

print('법인등록번호:', '조회 성공' if 결과['법인등록번호'] else '확인 필요')
print('종목코드:', 결과['종목코드'] or '확인 필요')
print('재무 행수:', len(재무표), '| 주가 행수:', len(주가표), '| 뉴스 건수:', len(뉴스표))
for 경고 in 결과['경고']:
    print('확인 필요:', 경고)

print('\n[재무정보]')
display(재무표)
print('\n[주식시세 최근 5건]')
display(주가표.tail(5))
print('\n[뉴스정보 최근 5건]')
display(뉴스표.head(5))

## 3단계 — pandas 결측·중복 점검 / 통계 집계

In [ ]:
print('결측·중복 점검')
display(결과['품질'])
print('기초 통계 / 집계')
display(결과['통계'])
print('데이터 출처와 시세 기준일')
display(결과['출처'])
print('원자료 기반 기계적 사실 검증')
display(결과['사실검증'])

## 4단계 — 차트 만들기

데이터가 확보된 경우에만 그립니다. 결측값을 임의 생성하지 않습니다.

In [ ]:
# 재무는 기존 matplotlib 그래프를 유지합니다.
import matplotlib.pyplot as plt
from stock_chart import (prepare_stock_history, select_stock_period,
                         summarize_stock_period, make_stock_figure)

if not 재무표.empty:
    그림 = 재무표.sort_values('연도').set_index('연도')
    (그림[['매출액', '영업이익', '당기순이익']] / 100000000).plot(kind='bar', figsize=(9, 4))
    plt.title('Financial trends (100 million KRW)')
    plt.ylabel('100 million KRW')
    plt.tight_layout()
    plt.show()
else:
    print('재무 차트: 데이터 없음 (확인 필요)')

# 주가는 실제 최신 거래일을 기준으로 기간을 자르고 같은 X축으로 시세와 거래량을 표시합니다.
주가이력 = prepare_stock_history(주가표)
선택기간 = '3개월'   # '1개월', '3개월', '6개월', '1년', '전체' 중 변경할 수 있습니다.
주가분석표 = select_stock_period(주가이력, 선택기간)

if not 주가분석표.empty:
    기술통계 = summarize_stock_period(주가이력, 주가분석표)
    print('분석 기준일:', 기술통계['기준일'].strftime('%Y-%m-%d'))
    print('최근 종가:', f"{기술통계['최근종가']:,.0f}원")
    print('기간 수익률:', f"{기술통계['선택기간수익률(%)']:+.2f}%" if pd.notna(기술통계['선택기간수익률(%)']) else '확인 필요')
    차트 = make_stock_figure(주가분석표, chart_type='캔들', moving_averages=(5, 20, 60))
    display(차트)
else:
    print('주가 차트: 데이터 없음 (확인 필요)')


### 별도 검증 셀 — 캔들차트·이동평균선·날짜축

실제 수집된 주가가 없으면 차트 API 검증은 **테스트 예정**으로 표시합니다. 인증키 없이 실행하는 모의 테스트는 마지막 단계를 사용하세요.

In [ ]:
# 이 셀을 앞 차트 셀과 분리하여 독립적으로 검증합니다.
if 주가분석표.empty:
    print('주가 실데이터가 없어 차트 실데이터 검증은 테스트 예정')
else:
    assert 주가분석표['날짜'].is_monotonic_increasing, '날짜 오름차순 확인 필요'
    assert 주가분석표['날짜'].max() == 주가이력['날짜'].max(), '마지막 거래일 기준 조회 확인 필요'
    assert not (주가분석표['날짜'] > 주가이력['날짜'].max()).any(), '미래 날짜 포함 오류'
    assert 'SMA20' in 주가이력.columns
    if len(주가이력) >= 20:
        실제20일평균 = 주가이력['종가'].tail(20).mean()
        assert abs(주가이력.iloc[-1]['SMA20'] - 실제20일평균) < 1e-8
    assert 차트.layout.xaxis.type == 'category', '거래일 범주축이 아닙니다.'
    assert 차트.layout.yaxis2.range[0] == 0, '거래량 축 최솟값이 0이 아닙니다.'
    assert 차트.layout.xaxis.rangeslider.visible is False
    print('실제 주가 차트 기간·이동평균·축 설정 검증 통과!')

## 4-2단계 — 뉴스 관련성·중복 제거 (OpenAI 호출 없음)

검색 결과 중 **기업명이 직접 언급되거나 관련성이 추정되는 기사**만 분석 대상으로 남깁니다. 뉴스 원문은 조회하지 않습니다.

In [ ]:
from news_analysis import (
    prepare_news, analyze_news_articles, sentiment_summary, verify_news_analysis
)

print('기업 관련 뉴스 후보:', len(결과['뉴스']), '건')
display(결과['뉴스'][['기사ID', '발행시각(KST)', '제목', '언론사', '사전관련성', '링크']])
print('제외된 뉴스:', len(결과['뉴스제외']), '건')
display(결과['뉴스제외'][['제목', '제외사유']].head(10))

### 별도 검증 셀 — 뉴스 필터링 (실제 API와 무관한 모의 샘플도 검사)

In [ ]:
# 모의 데이터에서도 기업 관련 없는 뉴스를 제거하는지 검증
가상뉴스 = pd.DataFrame([
    {'발행일':'2026-10-20T00:00:00Z', '제목':'삼성전자 신규 제품 발표',
     '언론사':'시험뉴스', '요약':'삼성전자가 새 제품을 발표', '링크':'https://example.com/a', '출처':'검증'},
    {'발행일':'2026-10-20T00:00:00Z', '제목':'래블업 신규 투자 유치',
     '언론사':'시험뉴스', '요약':'GPU 기업 투자 소식', '링크':'https://example.com/b', '출처':'검증'},
])
가상후보, 가상제외 = prepare_news(가상뉴스, '삼성전자')
assert len(가상후보) == 1, '삼성전자 관련 기사 한 건만 남아야 합니다.'
assert len(가상제외) == 1, '기업 무관 기사는 제외되어야 합니다.'
assert '삼성전자' in 가상후보.iloc[0]['제목']
assert 결과['뉴스']['기사ID'].is_unique, '실제 조회 뉴스의 ID 중복이 있습니다.'
print('뉴스 관련성 필터 검증 통과')

## 4-3단계 — 기사별 OpenAI 긍정·부정 영향 분석

**주의:** 아래 셀은 실제 API를 호출하므로 요금이 발생할 수 있습니다. 긍정·부정은 기사 제목/검색 요약으로 추정한 **기업에 대한 영향 방향**이며 기사 진위나 주가 예측 결과가 아닙니다.

분류: 긍정 / 부정 / 중립 / 혼합 / 판단보류. 기사별 영향분야·위험요인·판단근거가 추가됩니다.

In [ ]:
뉴스분석표 = pd.DataFrame()
if not OpenAI키 or OpenAI키.startswith('발급받은_'):
    print('OpenAI 키 미설정: 기사별 AI 분류는 테스트 예정')
elif 결과['뉴스'].empty:
    print('기업 관련 기사 없음: 기사별 AI 분류는 테스트 예정')
else:
    뉴스분석표 = analyze_news_articles(
        결과['뉴스'], company=기업명, openai_key=OpenAI키, model=모델
    )
    결과['뉴스분석'] = 뉴스분석표
    결과['뉴스감성검증'] = verify_news_analysis(결과['뉴스'], 뉴스분석표)
    print('기사별 감성 분석 결과')
    display(뉴스분석표[['기사ID', '제목', '감성', 'AI관련성', '영향분야', '위험요인', '판단근거']])
    print('기사 감성 집계')
    display(sentiment_summary(뉴스분석표))
    display(결과['뉴스감성검증'])

### 별도 검증 셀 — AI 뉴스 분류 결과 ID·링크·레이블 일치 확인

In [ ]:
if 뉴스분석표.empty:
    print('OpenAI 기사별 분류 미실행: 실제 분석 검증은 테스트 예정')
else:
    assert len(뉴스분석표) == len(결과['뉴스']), '분류된 기사 수가 원본과 다릅니다.'
    assert 뉴스분석표['기사ID'].is_unique, '기사 ID가 중복됩니다.'
    assert 뉴스분석표['감성'].isin(['긍정','부정','중립','혼합','판단보류']).all(), '알 수 없는 감성 분류가 있습니다.'
    assert 뉴스분석표['링크'].tolist() == 결과['뉴스']['링크'].tolist(), '원문 기사 링크가 변경되었습니다.'
    검증 = verify_news_analysis(결과['뉴스'], 뉴스분석표)
    assert (검증.iloc[:3]['상태'] == '확인').all(), '뉴스 분석 구조적 검증 실패'
    print('뉴스 감성 분류/원문 링크 검증 통과. 기사 사실 여부는 별도 확인 필요.')

## 5단계 — Excel 파일 저장

재무·주가·뉴스·품질·통계·검증·출처를 별도 시트로 저장합니다.

In [ ]:
엑셀경로 = Path(f'{기업명}_통합분석.xlsx')
엑셀경로.write_bytes(excel_bytes(결과))
print('Excel 저장 완료:', 엑셀경로.resolve())

### 별도 검증 셀 — 실제 API 수집 / Excel 검증

인증/승인 오류로 재무나 시세가 조회되지 않았으면 **실패**하는 것이 올바릅니다. 실데이터 없이 통과했다고 표시하지 않습니다.

In [ ]:
from openpyxl import load_workbook

assert 결과['기업명'] == 기업명
assert 결과['법인등록번호'] and len(결과['법인등록번호']) == 13, '기업기본정보 API 확인 필요'
assert isinstance(재무표, pd.DataFrame) and not 재무표.empty, '재무 API 실데이터 확인 필요'
assert isinstance(주가표, pd.DataFrame) and not 주가표.empty, '금융위원회 또는 대체 주가 실데이터 확인 필요'
assert (주가표['종가'].notna()).all(), '종가 결측값 확인 필요'
assert {'종가', '거래량', '날짜', '출처'}.issubset(set(주가표.columns))
assert not 결과['품질'].empty
assert 엑셀경로.is_file() and 엑셀경로.stat().st_size > 1000

# 공식 금융위원회 시세 API 자체의 연동 성공까지 확인하고 싶으면 아래 줄을 활성화하세요.
# assert 주가표['출처'].str.contains('금융위원회').all(), '공식 주식시세 API 조회 실패 - 대체 출처 사용 중'

검증엑셀 = load_workbook(엑셀경로, read_only=True)
assert {'재무정보', '주가정보', '뉴스정보', '데이터품질', '통계집계', '사실검증', '데이터출처', '차트'}.issubset(set(검증엑셀.sheetnames))
assert 검증엑셀['주가정보'].max_row == len(주가표) + 1
검증엑셀.close()
print('실제 조회 결과와 Excel 검증 통과!')

## 6단계 — OpenAI 투자검토 보고서 생성

분석 수치는 pandas가 계산하며, AI는 수익성·안정성·추세·뉴스 위험 **정성 검토**를 작성합니다. OpenAI API 호출에 비용이 발생할 수 있습니다.

In [ ]:
if not OpenAI키 or OpenAI키.startswith('발급받은_'):
    print('OpenAI API 키 미설정: 보고서 생성은 건너뜁니다.')
    AI보고서 = None
else:
    AI보고서 = ai_review(결과, OpenAI키, model=모델)
    for key, 제목 in [('profitability', '수익성'), ('stability', '안정성'),
                     ('trend', '주가 추세'), ('news_risks', '뉴스 위험 요인'),
                     ('conclusion', '종합 검토')]:
        print('\n[' + 제목 + ']')
        print(AI보고서[key])
    보고서경로 = Path(f'{기업명}_투자검토보고서.md')
    보고서경로.write_text(report_markdown(결과, AI보고서), encoding='utf-8')
    AI엑셀경로 = Path(f'{기업명}_AI투자검토_통합보고서.xlsx')
    AI엑셀경로.write_bytes(excel_bytes(결과, AI보고서))
    print('투자검토 보고서 저장:', 보고서경로, AI엑셀경로)

### 별도 검증 셀 — OpenAI 보고서 수치 차단 검사

In [ ]:
if AI보고서 is None:
    print('OpenAI 인증키 미설정: AI 보고서 검증은 테스트 예정')
else:
    필수 = ['profitability', 'stability', 'trend', 'news_risks', 'conclusion']
    assert all(isinstance(AI보고서.get(k), str) and AI보고서[k].strip() for k in 필수)
    assert all(not re.search(r'[0-9０-９]', AI보고서[k]) for k in 필수), 'AI 정성 의견에 수치가 포함되었습니다.'
    print('AI 보고서 형식 및 숫자 포함 여부 검사 통과')
    print('주의: AI 의견의 진위와 뉴스 기사 진위를 보증하는 검사는 아닙니다.')

## 7단계 — 인증키 없이 자동 검증하기

모의 응답을 이용하는 단위 테스트입니다. API 실서버 테스트와는 구분합니다.

In [ ]:
import unittest
from pathlib import Path
if Path('test_core.py').exists() and Path('test_stock_chart.py').exists():
    suite = unittest.defaultTestLoader.discover('.', pattern='test_*.py')
    결과검증 = unittest.TextTestRunner(verbosity=2).run(suite)
    assert 결과검증.wasSuccessful(), '단위 테스트 실패'
else:
    print('test_core.py와 test_stock_chart.py를 같은 폴더에 넣어주세요.')

## 사용 시 주의

- 주식시세정보 API 자료는 비실시간·일별 갱신입니다. 화면의 기준일을 반드시 확인합니다.
- Google 뉴스 RSS·NewsAPI·SerpAPI 검색 결과는 기사 제목/요약이며 사실 확정 자료가 아닙니다.
- 사업연도별 재무정보가 제공되지 않으면 임의 숫자를 채우지 않습니다.
- 공공데이터 주식시세의 **공공누리 4유형 및 제3자 재배포 제한**을 확인한 뒤 웹 배포하세요.
- `.env`와 인증키는 GitHub에 업로드하거나 제출하지 마세요.